In [7]:
from pathlib import Path

import pandas as pd
import requests
from google.cloud import bigquery
from google.oauth2 import service_account

In [8]:
BQ_PROJECT_ID = "gedas-analytics-dev"
BQ_DATASET_ID = "external__google_sheets"
BQ_TABLE_ID = "stocks_cost_basis"
CREDENTIALS_PATH = Path(r"d:/Coding/DAE/Airflow/credentials/gc-gedas-analytics-dev-sa.json")

credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH,
    scopes=["https://www.googleapis.com/auth/bigquery"],
)
client = bigquery.Client(project=BQ_PROJECT_ID, credentials=credentials)
symbols_df = client.query(
    f"""
    SELECT DISTINCT ticker AS symbol
    FROM `{BQ_PROJECT_ID}.{BQ_DATASET_ID}.{BQ_TABLE_ID}`
    WHERE ticker IS NOT NULL
    """
).to_dataframe()
symbols = symbols_df["symbol"].tolist()

c:\Users\gedim\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [9]:
TWELVE_DATA_API_KEY = "a9a6b890ce244303b9cb2a5aabd55211"

In [10]:
# One ticker only on the free plan (1 credit). Do not loop symbols yet.
# Re-run this cell only when you need a new API response.
response = requests.get(
    "https://api.twelvedata.com/eod",
    params={"symbol": "AAPL", "apikey": TWELVE_DATA_API_KEY},
)
payload = response.json()
payload

{'symbol': 'AAPL',
 'exchange': 'NASDAQ',
 'mic_code': 'XNGS',
 'currency': 'USD',
 'datetime': '2026-10-20',
 'timestamp': 1792592340,
 'close': '339.5'}

In [11]:
df = pd.DataFrame(
    [
        {
            "date": payload["datetime"],
            "symbol": payload["symbol"],
            "close": float(payload["close"]),
            "currency": payload["currency"],
            "exchange": payload["exchange"],
        }
    ]
)
df

,date,symbol,close,currency,exchange
0,2026-10-20,AAPL,339.5,USD,NASDAQ
